<a href="https://colab.research.google.com/github/sanyamsh7/Agentic-RAG-Pipeline-From-Scratch/blob/main/Module5_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 5 -  LANGUAGE MODELS AND GENERATION

Core Concepts explored:

1. Understanding LLMs:

    - How they predict next tokens
    - Different architectures (GPT, T5, BERT)
    - Why Flan-T5 is perfect for RAG


2. LLM Parameters Demystified:

    - temperature: 0.3  ← Factual (RAG)
    - temperature: 0.7  ← Balanced
    - temperature: 1.5  ← Creative (might hallucinate!)
    - max_new_tokens: 150-200  ← Sweet spot for RAG

3. Prompt Engineering for RAG:

    - Prompt structure
    - With vs. without context comparison
    - Constraining to prevent hallucinations

4. Complete RAG pipeline:
    - User query > Embed > Search Vector DB > Retrieve Chunks > Build Prompt > LLM Generate > Answer + Citations

5. Production-Ready System:
    - Complete ProductionRAG class
    - Best practices built-in
    - Source citations included

In [1]:
# installing libraries
!pip install -q --upgrade pip
!pip install -q langchain langchain-community langchain-chroma
!pip install -q transformers torch sentence-transformers
!pip install -q chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.27.1 requ

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.language_models.llms import LLM
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import Any, List, Optional, Dict
from pydantic import Field
import warnings
warnings.filterwarnings('ignore')


## Core Concept: Token Prediction
LLMs predict the next token (word/subword) in a sequence.

Example:

Input:  "The sky is"

Output: "blue" (most probable next word)

They learn this by training on BILLIONS of words from:
- Books and articles
- Websites and code
- Conversations and documents

THREE MAIN ARCHITECTURES:
------------------------

1. DECODER-ONLY (GPT-style)

   How it works: Takes input, predicts next tokens

   Examples: GPT-3, GPT-4, LLaMA, Mistral

   Best for: Text generation, completion, chat
   
   Flow: "Hello" → [Model] → "world, how are you?"

2. ENCODER-DECODER (T5-style) ← WE USE THIS

   How it works: Encoder understands, decoder generates

   Examples: T5, BART, Flan-T5

   Best for: Translation, summarization, Q&A
   
   Flow: "Translate: Hello" → [Encoder→Decoder] → "Bonjour"

3. ENCODER-ONLY (BERT-style)

   How it works: Understands and represents text

   Examples: BERT, RoBERTa

   Best for: Embeddings, classification, understanding
   
   Flow: "I love AI" → [Encoder] → [0.2, 0.8, ..., 0.5]

WHY FLAN-T5 FOR RAG?
-------------------
✅ Instruction-following (trained to follow commands)

✅ Compact size (runs on CPU, no GPU needed)

✅ Great at Q&A tasks

✅ Encoder-decoder = better for reading context + answering

✅ Open-source and free

In [3]:
class FlanT5LLM(LLM):
  """
  Production grade Flan-T5 wrapper with anti-hallucination features

  key features:
  1. low temp. for factual answers
  2. proper tokenization handling
  3. deterministic decoding options
  4. integration with langchain
  """
  model_name: str = Field(default = "google/flan-t5-base")
  model: Any = Field(default = None, exlude = True)
  tokenizer: Any = Field(default = None, exlude = True)
  max_length: int = Field(default = 200)
  temperature: float = Field(default = 0.1) # low for factual
  top_p: float = Field(default = 0.9)
  repetition_penalty: float = Field(default = 1.2) # prevents repetition

  class Config:
    arbitrary_types_allowed = True

  def __init__(self, **kwargs):
    super().__init__(**kwargs)

    print(f"loading : {self.model_name}")
    print(f"max length: {self.max_length}")
    print(f"temperature: {self.temperature}")

    # loading tokenizer and model
    self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
    self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

    print("model loaded successfully")
    print(f"model size : {sum(p.numel() for p in self.model.parameters()) / 1e6:.1f}M parameters\n")

  @property
  def _llm_type(self)-> str:
    return "flan-t5"

  def _call(
      self, prompt: str,
      stop : Optional[List[str]] = None,
      run_manager: Optional[Any] = None, **kwargs: Any) -> str:
      """
      generate text with anti-hallucination measures

      anti-hallucination techniques:
      1. low temperature (0.1-0.3) for factual consistency
      2. repetition penalty to avoid loops
      3. proper max_length to prevent cutoffs
      4. top-p sampling for quality control
      """
      # tokenizing with proper truncation
      inputs = self.tokenizer(
          prompt,
          return_tensors = "pt",
          max_length = 512,
          truncation = True,
          padding = False
      )

      # generate with anti-hallucination settings
      with torch.no_grad():
        outputs = self.model.generate(
            inputs.input_ids,
            max_length = self.max_length,
            temperature = self.temperature,
            do_sample = True if self.temperature > 0 else False,
            top_p = self.top_p,
            repetition_penalty = self.repetition_penalty,
            num_beams = 1, # greedy for low temp
            early_stopping = True,
            num_return_sequences = 1
        )

        # decode cleanly
        result = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens = True,
            clean_up_tokenization_spaces = True
        )

        return result.strip()

  @property
  def _identifying_params(self) -> Dict[str, Any]:
    return {
        "model_name": self.model_name,
        "max_length": self.max_length,
        "temperature": self.temperature,
        "top_p": self.top_p,
        "repetition_penalty": self.repetition_penalty
    }


## ⚙️  CRITICAL PARAMETERS FOR RAG

1. TEMPERATURE (Most Important!)
   --------------------------------
   Controls randomness in token selection
   
   temperature = 0.0:
     - Deterministic (always same answer)
     - Picks highest probability token
     - Use for: Math, facts, code
     - Risk: Can be repetitive
   
   temperature = 0.1-0.3: ⭐ BEST FOR RAG
     - Mostly deterministic
     - Slight variation allowed
     - Use for: Q&A, factual tasks
     - Sweet spot: 0.1 for facts, 0.3 for summaries
   
   temperature = 0.5-0.7:
     - Balanced creativity
     - Use for: General chat, explanations
     - Risk: May introduce errors
   
   temperature = 1.0+:
     - Very creative/random
     - Use for: Story writing, brainstorming
     - Risk: High hallucination rate
   
   FOR RAG: ALWAYS USE 0.1-0.3!

2. MAX_LENGTH / MAX_NEW_TOKENS
   -----------------------------
   Maximum tokens to generate
   
   Too low (50):   Answers cut off mid-sentence

   Sweet spot (150-200): ⭐ Complete answers

   Too high (500+): Rambling, repetition
   
   FOR RAG: 150-200 tokens

3. TOP_P (Nucleus Sampling)
   -------------------------
   Consider tokens until cumulative probability = p
   
   top_p = 0.9: ⭐ RECOMMENDED

     - Consider top 90% of probability mass
     - Filters out unlikely tokens
     - Good balance
   
   top_p = 1.0: No filtering (more random)

   top_p = 0.5: Very conservative

4. REPETITION_PENALTY
   -------------------
   Penalizes repeating tokens
   
   1.0: No penalty

   1.2: ⭐ RECOMMENDED - prevents loops

   1.5+: Too aggressive, unnatural text
   
5. DO_SAMPLE
   ----------
   True: Sample from distribution (use with temp > 0)

   False: Greedy decoding (always most likely)
   
   FOR RAG: True with low temperature

RECOMMENDED SETTINGS FOR RAG:
----------------------------
temperature: 0.1-0.3  ← CRITICAL!

max_length: 150-200

top_p: 0.9

repetition_penalty: 1.2

do_sample: True

EXPERIMENTATION WITH DIFFERENT VALUES IS STILL REQUIRED!

In [22]:
# initialize LLM with optimal settings
llm = FlanT5LLM(
    model_name = "google/flan-t5-large",
    max_length = 200,
    temperature = 0.1, # after experimenting with different values below
    top_p = 0.3, # works better for me
    repetition_penalty = 1.2,
)

In [21]:
# testing different temperatures and top_p values

question = "What is the capital of France?"
print("question: ", question)
temperature = [0.0, 0.1, 0.5, 1.0, 1.5]
p = [0.1,0.3, 0.3, 0.4, 0.5]

for prob, temp in zip(p, temperature):
  print("temperature", temp)
  # create temporary llm with this temperature
  temp_llm = FlanT5LLM(
      model_name = "google/flan-t5-large",
      max_length = 50,
      temperature = temp,
      top_p = prob,
  )
  answer = temp_llm.invoke(question)
  print("answer:", answer)
  print()

## Prompt Engineering for RAG

📝 THE ANATOMY OF A GOOD RAG PROMPT

BAD PROMPT (Causes Hallucinations):
-----------------------------------
"Question: What is machine learning?
Answer:"

Problems:
❌ No context provided
❌ No constraints
❌ Model can make things up
❌ No instruction to stay grounded

GOOD PROMPT (Prevents Hallucinations):
--------------------------------------
"Use ONLY the following context to answer the question.
If the answer is not in the context, say 'I don't have enough information.'
Do not make up information.

Context:
[Retrieved document text]

Question: What is machine learning?

Answer:"

Why it works:
✅ Explicit instruction to use context
✅ Clear constraint (ONLY use context)
✅ Escape hatch for unknown answers
✅ Prohibits making things up

KEY ELEMENTS OF RAG PROMPTS:
---------------------------
1. INSTRUCTION: "Use the following context..."
2. CONSTRAINT: "ONLY", "If not in context, say so"
3. CONTEXT: Retrieved document chunks
4. QUESTION: User's query
5. FORMAT: "Answer:" to guide output

ADVANCED TECHNIQUES:
-------------------
- Add "Be concise" to prevent rambling
- Add "Cite sources" for transparency
- Add "Step by step" for complex questions
- Use examples (few-shot prompting)

In [6]:
# create knowledge base
documents = [
    Document(
        page_content="""Machine learning is a subset of artificial intelligence
        that enables systems to learn and improve from experience without being
        explicitly programmed. It focuses on developing algorithms that can access
        data and learn from it. Common applications include image recognition,
        natural language processing, and predictive analytics.""",
        metadata={"source": "ml_intro.pdf", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""Neural networks are computing systems inspired by biological
        neural networks that constitute animal brains. They consist of layers of
        interconnected nodes (neurons) that process information using a connectionist
        approach to computation. Each connection can transmit a signal from one neuron
        to another, and the receiving neuron processes it.""",
        metadata={"source": "neural_nets.pdf", "page": 3, "topic": "Neural Networks"}
    ),
    Document(
        page_content="""Deep learning is a subset of machine learning that uses
        neural networks with multiple layers (deep neural networks). These networks
        can learn hierarchical representations of data, making them powerful for
        complex tasks like image recognition, speech processing, and natural language
        understanding. Deep learning has revolutionized AI in the last decade.""",
        metadata={"source": "deep_learning.pdf", "page": 5, "topic": "Deep Learning"}
    ),
    Document(
        page_content="""Python is a high-level, interpreted programming language
        known for its simplicity and readability. It's widely used in data science,
        machine learning, web development, and automation. Python has a rich ecosystem
        of libraries including NumPy for numerical computing, Pandas for data analysis,
        and TensorFlow and PyTorch for machine learning.""",
        metadata={"source": "python_guide.pdf", "page": 2, "topic": "Python"}
    ),
    Document(
        page_content="""The Transformer architecture, introduced in 2017, revolutionized
        natural language processing. It uses self-attention mechanisms to process
        sequences in parallel, unlike previous sequential models. Transformers are
        the foundation of modern language models like GPT, BERT, and T5.""",
        metadata={"source": "transformers.pdf", "page": 7, "topic": "Transformers"}
    ),
]

print("Created knowledge base with :", len(documents), "documents")
print("Topics:", set([doc.metadata["topic"] for doc in documents]))


Created knowledge base with : 5 documents
Topics: {'Transformers', 'Python', 'Deep Learning', 'ML', 'Neural Networks'}


In [20]:
# set up vector store
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2",
    model_kwargs = {"device": "cuda"},
    encode_kwargs = {"normalize_embeddings": True}
)

vectorstore = Chroma.from_documents(
  documents=documents,
  embedding = embeddings,
  collection_name = "production_rag"
)

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 2}
)
print(f"Vector store created with {len(documents)} documents")
print(f"Retriever configured to return top 2 results\n")

## Building the complete RAG pipeline

🔗 RAG PIPELINE FLOW:

```
User Question
    ↓
[1] EMBED QUESTION
    → Convert to 384D vector
    ↓
[2] SEARCH VECTOR DB
    → Find top-k similar chunks
    ↓
[3] RETRIEVE CHUNKS
    → Get relevant documents
    ↓
[4] FORMAT CONTEXT
    → Combine chunks into string
    ↓
[5] BUILD PROMPT
    → Context + Question + Instructions
    ↓
[6] LLM GENERATE
    → Answer based on context
    ↓
[7] PARSE OUTPUT
    → Clean string output
    ↓
Answer + Citations
```

In [9]:
def format_docs(docs: List[Document]) -> str:
  formatted =[]
  for i, doc in enumerate(docs):
    formatted.append(f"[Source {i}: {doc.metadata['source']}, Page {doc.metadata['page']}]\n{doc.page_content}")
  return "\n\n".join(formatted)

In [10]:
# create anti-hallucination prompt template
template = """You are a helpful assistant that answers questions using ONLY the provided context.

IMPORTANT RULES:
1. Use ONLY information from the context below
2. If the answer is not in the context, say "I don't have enough information to answer that question."
3. Do not make up or infer information
4. Be concise and accurate
5. Cite the source number when possible (e.g., "According to Source 1...")

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

# build lcel chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [11]:
# testing rag chain

test_questions = [
    "what is machine learning?",
    "how do neural networks work?",
    "what programming language is good for AI?",
    "what is the capital of France?" # not in knowledge base
]

for question in test_questions:
  print("question: ", question)
  answer = rag_chain.invoke(question)
  print("answer: ", answer)

  # show source
  sources = retriever.invoke(question)
  print("sources used: ")
  for i, doc in enumerate(sources):
    print(f"{i}. {doc.metadata['source']} (page {doc.metadata['page']}) - {doc.metadata['topic']}")


question:  what is machine learning?
answer:  Artificial intelligence
sources used: 
0. ml_intro.pdf (page 1) - ML
1. deep_learning.pdf (page 5) - Deep Learning
question:  how do neural networks work?
answer:  They consist of layers of interconnected nodes (neurons) that process information using a connectionist approach to computation.
sources used: 
0. neural_nets.pdf (page 3) - Neural Networks
1. deep_learning.pdf (page 5) - Deep Learning
question:  what programming language is good for AI?
answer:  I don't have enough information to answer that question.
sources used: 
0. python_guide.pdf (page 2) - Python
1. ml_intro.pdf (page 1) - ML
question:  what is the capital of France?
answer:  I don't have enough information to answer that question.
sources used: 
0. python_guide.pdf (page 2) - Python
1. ml_intro.pdf (page 1) - ML


In [12]:
# comparison with vs without context
question = "what is deep learning?"
print("question: ", question)
print("Without RAG pure llm may hallucinate: ")
basic_prompt = f"Answer this question concisely: {question}"
no_rag_answer = llm.invoke(basic_prompt)
print("answer: ", no_rag_answer)

question:  what is deep learning?
Without RAG pure llm may hallucinate: 
answer:  deep learning (DPL) refers to incorporating deep knowledge in decision making by a machine learning method that uses supervised training techniques, i.e.


In [13]:
# with rag
rag_answer = rag_chain.invoke(question)
print("answer: ", rag_answer)

answer:  a subset of machine learning that uses neural networks with multiple layers (deep neural networks).


KEY INSIGHTS:
------------
WITHOUT RAG:
- Generic knowledge from training data
- May be outdated (training cutoff)
- Can't cite sources
- Higher hallucination risk

WITH RAG:
- Specific to YOUR documents
- Always up-to-date (just update docs)
- Provides source citations
- Grounded, verifiable answers

# Production-Ready RAG class

In [16]:
class ProductionRAG:
  """
  production-grade RAG system with best practices

  features:
    - anti-hallucination prompting
    - source citation
    - error handling
    - configurable parameters
    - langhchain + lcel integration
  """
  def __init__(
      self,
      documents: List[Document],
      embedding_model: str = "all-MiniLM-L6-v2",
      llm_model: str = "google/flan-t5-large",
      temperature: float = 0.1,
      retrieval_k: int = 2
  ):
      """
      Initialize production RAG system.
      args:
        - documents: knowledge base documents
        - embedding_model: embedding model name
        - llm_model: LLM model name
        - temperature: LLM temperature
        - retrieval_k: number of documents to retrieve
      """
      print("Initializing Production RAG System...")
      print(f"Documents: {len(documents)}")
      print(f"LLM: {llm_model}")
      print(f"Temperature: {temperature} (factual mode)")
      print(f"Retrieval: Top-{retrieval_k}")

      # setup embeddings
      self.embeddings = HuggingFaceEmbeddings(
          model_name = embedding_model,
          model_kwargs = {'device': 'cpu'},
          encode_kwargs = {'normalize_embeddings': True}
      )

      # setup vector space
      self.vectorstore = Chroma.from_documents(
          documents = documents,
          embedding = self.embeddings,
          collection_name = "production_rag_final"
      )

      # setup retreiver
      self.retriever = self.vectorstore.as_retriever(
          search_type = "similarity",
          search_kwargs = {"k": retrieval_k}
      )

      # setup LLM with anti-hallucination settings
      self.llm = FlanT5LLM(
          model_name = llm_model,
          max_length = 200,
          temperature = temperature,
          repetition_penalty = 1.2,
          top_p = 0.3,
      )

      # anti-hallucination prompt
      template = """You are a helpful AI assistant. Answer using ONLY the context provided

      RULES:
      1. Use ONLY the context below
      2. If answer not in context: "I don't have enough information."
      3. Be concise and accurate
      4. Cite sources when possible

      Context:
      {context}

      Question: {question}

      Answer:"""

      self.prompt = PromptTemplate.from_template(template)

      # build chain
      self.chain = (
          {
              "context": self.retriever | self._format_docs,
              "question": RunnablePassthrough()
          }
          | self.prompt
          | self.llm
          | StrOutputParser()
      )
      print("Production RAG initialized!\n")

  def _format_docs(self, docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
      source = doc.metadata.get("source", "Unknown")
      page = doc.metadata.get("page", "N/A")
      formatted.append(f"[Source {i}: {source}, Page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

  def ask(self, question: str) -> Dict[str, Any]:
    # generate answer
    answer = self.chain.invoke(question)

    # get source documents
    sources = self.retriever.invoke(question)

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
               "source": doc.metadata.get("source", "Unknown"),
                "page": doc.metadata.get("page", "N/A"),
                "topic": doc.metadata.get("topic", "General"),
                "excerpt": doc.page_content[:150] + "..."
            }
            for doc in sources
        ],
            "num_sources": len(sources)
    }

  def batch_ask(self, questions: List[str]) -> List[Dict[str, Any]]:
    return [self.ask(q) for q in question]



In [19]:
# testing production system
rag_system = ProductionRAG(
    documents=documents,
    temperature=0.1,  # Factual mode
    retrieval_k=2
)

test_questions = [
    "What are neural networks inspired by?",
    "What is the difference between machine learning and deep learning?",
    "Which libraries are used for machine learning in Python?",
]

In [18]:
for question in test_questions:
    result = rag_system.ask(question)

    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"\nSources ({result['num_sources']}):")
    for i, source in enumerate(result['sources'], 1):
        print(f"{i}. {source['source']} (Page {source['page']}) - {source['topic']}")
    print(f"Preview: {source['excerpt'][:100]}...")
    print("=" * 50)


Q: What are neural networks inspired by?
A: biological neural networks

Sources (2):
1. neural_nets.pdf (Page 3) - Neural Networks
2. deep_learning.pdf (Page 5) - Deep Learning
Preview: Deep learning is a subset of machine learning that uses
        neural networks with multiple layers...
Q: What is the difference between machine learning and deep learning?
A: Deep learning is a subset of machine learning

Sources (2):
1. deep_learning.pdf (Page 5) - Deep Learning
2. ml_intro.pdf (Page 1) - ML
Preview: Machine learning is a subset of artificial intelligence
        that enables systems to learn and im...
Q: Which libraries are used for machine learning in Python?
A: TensorFlow and PyTorch

Sources (2):
1. python_guide.pdf (Page 2) - Python
2. ml_intro.pdf (Page 1) - ML
Preview: Machine learning is a subset of artificial intelligence
        that enables systems to learn and im...


# FINAL: KEY TAKEAWAYS & BEST PRACTICES

1. UNDERSTANDING LLMs:

   ✅ Token prediction machines

   ✅ Different architectures for different tasks

   ✅ Flan-T5: Instruction-following, great for RAG

   ✅ Encoder-Decoder superior for context+question tasks

2. PREVENTING HALLUCINATIONS:

   ✅ Temperature: 0.1-0.3 (CRITICAL!)

   ✅ Repetition penalty: 1.2

   ✅ Explicit prompt constraints

   ✅ "If not in context, say so" instruction

   ✅ Source citation requirement

3. OPTIMAL PARAMETERS FOR RAG:

   temperature: 0.1 (factual)

   max_length: 150-200

   repetition_penalty: 1.2

   do_sample: True

4. PROMPT ENGINEERING:

   ✅ Clear instructions ("Use ONLY context")

   ✅ Explicit constraints

   ✅ Escape hatch for unknowns

   ✅ Request source citations

   ✅ Prohibit making things up

5. COMPLETE RAG PIPELINE:
```
   Query → Embed → Search → Retrieve → Format → Prompt → Generate → Parse
   ```
  Each step is critical:

   - Embed: Semantic search

   - Retrieve: Relevant context

   - Prompt: Anti-hallucination instructions

   - Generate: Low-temp factual output

6. PRODUCTION BEST PRACTICES:

   ✅ Use tested library versions

   ✅ Low temperature (0.1-0.3)

   ✅ Anti-hallucination prompts

   ✅ Source citations

   ✅ Error handling

   ✅ Configurable parameters

   ✅ Proper LCEL chains

7. LANGCHAIN + LCEL:

   ✅ Modular, composable chains

   ✅ Easy to debug and modify

   ✅ Production-ready
   
   ✅ Standard interface

CRITICAL SUCCESS FACTORS:
------------------------
1. Temperature = 0.1 for RAG (MOST IMPORTANT!)
2. Explicit "use only context" in prompt
3. Provide escape hatch for unknowns
4. Cite sources
5. Use proper LCEL chains

🎉 YOU NOW HAVE A PRODUCTION RAG SYSTEM!
========================================
- Prevents hallucinations ✅
- Provides citations ✅
- Handles unknowns gracefully ✅
- Scales to any knowledge base ✅
- Uses modern LangChain patterns ✅